# 14. The model app: a JupyterLab sidebar workbench

The pieces you have met so far -- loading (tutorial 1), the explorer
(tutorial 12), the requirements scoreboard (tutorial 13), save and the
API client (tutorial 2) -- compose into one JupyterLab **app**:
`longeron.app.open()` docks a panel into the LEFT sidebar (look for the
longeron monogram icon) from which you load models and launch the other
surfaces as main-area tabs.

Run this notebook inside JupyterLab (`pixi run lab`) for the real
experience; the `explorer` extra provides the docking (ipylab).
Everywhere else -- nbclient, VS Code, the docs build -- the SAME widget
renders inline in the cell output and the launchers build inline
widgets instead of docking tabs, so every cell below runs headless too.

In [ ]:
import longeron
from longeron import app

application = app.open()  # in JupyterLab: docks (and reveals) the sidebar panel
application.layout_strategy

## Load models

The sidebar's path field (plus its **Browse** listing -- JupyterLab has
no OS file dialogs, so browsing is a small server-side directory tree)
loads a `.sysml`/`.json` file or a whole directory. The **Connect to
API...** fold does the same against any Systems Modeling API server
(`longeron serve`, the OMG pilot, Flexo) through
`longeron.client.Client`: URL + optional bearer token, then project and
commit pickers.

Every affordance has a programmatic twin on the returned handle -- the
notebook automation surface used below.

In [ ]:
model = application.load_path("../examples/drone.sysml")
[(entry.origin, entry.source) for entry in application.entries]

## Launch tabs

Each model row carries its actions: **Explore** docks a model explorer
tab (tutorial 12), **Score** docks a requirements scoreboard tab
(tutorial 13; `drone.sysml`'s geometric installation requirements keep
its button live, while a model without requirement usages gets an
honestly grayed-out button with a tooltip that says why). Relaunching replaces the model's tab instead of stacking
a second one, exactly like `explore()` on its own.

In [ ]:
explorer = application.explore_model(model)

scored = longeron.loads(
    """
package ScoutMini {
    part sys;
    requirement mission {
        attribute weight : Real = 2.0;
        requirement coverage { attribute weight : Real = 3.0; }
        requirement endurance;
    }
}
""",
    source_name="scout mini",
)
application.add_model(scored, source="inline demo text")
board = application.scoreboard_model(scored)
type(board).__name__

## The selection seam

The app tracks what you are working on and broadcasts it -- the hook a
property inspector (or any other tool) attaches to without touching the
app's internals: `current_model` / `on_model_selected(cb)` follow the
model list and row clicks, and `current_element` /
`on_element_selected(cb)` follow the selection inside every app-launched
explorer and scoreboard tab.

In [ ]:
seen = []
application.on_element_selected(lambda element: seen.append(element.qualified_name))

explorer.select("Drone::QuadCopter")  # a tree click does the same
(application.current_element.qualified_name, seen)

## Save and push

**Save** writes a file-loaded model back to its source
(`longeron.export.save`); directory-merged and in-memory models need an
explicit save-as path, and API-loaded models get **Push** instead --
a commit-message prompt feeding `client.push_commit`. Closing a row
(`close_model`) only drops it from the list; launched tabs live on.

In [ ]:
import tempfile
from pathlib import Path

target = Path(tempfile.mkdtemp()) / "scout_mini.sysml"
application.save_model(scored, path=target)  # save-as (a text-origin model)
application.close_model(scored)
(target.exists(), [entry.source for entry in application.entries])

Where to go next: tutorial 12 tours the explorer the app launches;
tutorial 13 the scoreboard; tutorial 2 the save/export surfaces the Save
button rides; and `docs/design/openmbee-integration.md` the API servers
the Connect fold talks to.

## Inspecting and editing

The app pairs its selection seam with an **item inspector**
(`longeron.inspector`, built by `app.open()` and exposed as
`application.inspector`). In JupyterLab it lives in the **right
sidebar -> the Inspector tab** (the box-and-lens icon), the Lab-native
home for property sheets; the first element you select in an
app-launched tab reveals it once (`app.open(reveal_inspector=False)`
disables that), after which it stays one click away. Inline it renders
wherever you display it. Click any element in an app-launched explorer
or scoreboard tab and the sheet follows: a kind chip + qualified-path header, read-only
rows (typings, multiplicity, direction, relationship ends -- the ends are
clickable and navigate the selection), and editable **name** /
**documentation** / **value** fields that commit on Enter/blur through
`longeron.edit`. A rename cascades into every textual reference or is
*honestly refused* -- the inline error strip lists exactly the references
that cannot be rewritten safely, and the field reverts.

In [ ]:
explorer.select("Drone::QuadCopter")  # the sheet follows any selection
application.inspector  # docked right in JupyterLab; renders inline here

In [ ]:
# every longeron.edit operation on a loaded model reaches the app's chrome:
# the model row grows a dirty dot (its tooltip lists the unsaved changes),
# launched explorer trees refresh on renames, and Save/Push enable until
# edit.track(model) is marked saved (which Save and Push do on success)
from longeron import edit

edit.set_doc(model, "Drone::QuadCopter", "The baseline quad-rotor airframe.")
tracker = edit.track(model)
(tracker.dirty, [(change.op, change.qname) for change in tracker.changes])

In [ ]:
# the inspector is wired end to end: it exists, and it received the
# selection the explorer fed through the app's seam (cell 12's
# `explorer.select(...)` -- a tree click in JupyterLab does the same)
assert application.inspector is not None
assert application.inspector.element is application.current_element
application.inspector.element.qualified_name